In [2]:
import numpy as np
import pandas as pd
!pip install xgboost
!pip install tensorflow

import tensorflow as tf




In [3]:
pd.read_csv('snote.csv')



,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,1162,85,0,159,0,0,13567,56581,21,1,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,3093,85,1283,317,2,2,13568,4023,0,2,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,1603,85,1266,175,9,2,13571,4,1,1,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,4275,85,1124,318,2,2,13567,3442,0,2,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,4333,85,1194,318,2,2,13570,3567,0,2,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
386741,794,28,65407,111,53,17,3744,15125,0,2,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5
386742,731,28,54705,111,53,17,644,15225,0,2,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5
386743,50,28,49164,19,443,6,2160,4796722,1,1,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5
386744,712,28,51691,111,53,17,571,12714,0,2,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5


In [4]:
df = pd.read_csv('snote.csv')

In [5]:

# Extract features (all columns except the target variable)
X = df.drop(columns=['Label'])

# Extract target variable (label)
y = df['Label']

In [6]:
from sklearn.model_selection import train_test_split

# Split the dataset into training (70%) and test (30%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Further split the training set into training (70%) and validation (30%) sets
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.3, random_state=42)

# Now have to X_train, X_val, X_test for features and y_train, y_val, y_test for labels


In [7]:
# 4 NEURAL NETWORKS

from sklearn.neural_network import MLPClassifier

mlp_classifier = MLPClassifier(hidden_layer_sizes=(100, ), max_iter=1000, random_state=42)

mlp_classifier.fit(X_train, y_train)

# y_pred = mlp_classifier.predict(X_test)


MLPClassifier(max_iter=1000, random_state=42)

In [8]:
# Make predictions for each classifier

y_pred_nn = mlp_classifier.predict(X_test)



In [11]:
# Compute confusion matrix for each classifier

from sklearn.metrics import confusion_matrix

cm_nn = confusion_matrix(y_test, y_pred_nn)
print(cm_nn)


[[47873     0     0     1     0     0]
 [    0    27     0 12475     1     3]
 [    0     0     1 13188     0     0]
 [    3     7     0 14287     2     3]
 [    0     1     0  7250  7442     4]
 [    0     5     0 12405     0  1046]]


In [26]:
# Compute TPR, TNR, FPR, FNR for each classifier using confusion matrix
def compute_metrics(cm):
    TP = cm[1, 1]
    TN = cm[0, 0]
    FP = cm[0, 1]
    FN = cm[1, 0]

    tpr = TP / (TP + FN)
    tnr = TN / (TN + FP)
    fpr = FP / (FP + TN)
    fnr = FN / (FN + TP)

    return tpr, tnr, fpr, fnr

# Compute metrics for each classifier

tpr_nn, tnr_nn, fpr_nn, fnr_nn = compute_metrics(cm_nn)


(1.0, 1.0, 0.0, 0.0)


In [12]:
from sklearn.metrics import accuracy_score

# Compute accuracy for each classifier

accuracy_nn = accuracy_score(y_test, y_pred_nn)

In [13]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Compute precision, recall, and F1 score for each classifier

precision_nn = precision_score(y_test, y_pred_nn, average='weighted')
recall_nn = recall_score(y_test, y_pred_nn, average='weighted')
f1_nn = f1_score(y_test, y_pred_nn, average='weighted')


In [14]:
# Print computed metrics for each classifier


print("Neural Network Metrics:")
print("Precision:", precision_nn)
print("Recall:", recall_nn)
print("F1 Score:", f1_nn)
print()



Neural Network Metrics:
Precision: 0.8700722188291231
Recall: 0.6091498310694339
F1 Score: 0.5626111738371394



In [15]:
!pip install dice-ml


In [16]:
from dice_ml import Dice
from dice_ml.utils import helpers # helper functions

In [17]:

continuous_features = df.select_dtypes(include=['float64', 'int64']).columns.tolist()

print("Continuous feature names:", continuous_features)

Continuous feature names: ['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size 

In [18]:
continuous_features =  ['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Blk Rate Avg', 'Subflow Fwd Pkts', 'Subflow Fwd Byts', 'Subflow Bwd Pkts', 'Subflow Bwd Byts', 'Init Fwd Win Byts', 'Init Bwd Win Byts', 'Fwd Act Data Pkts', 'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min']
outcome_name = 'Label'

In [19]:
# # Initialize DiCE explainer for a specific model (Gradient booosting machine)

##from dice_ml.utils import helpers

##d = dice_ml.Data(dataframe= df, continuous_features=continuous_features, outcome_name=outcome_name)

 ##data_interface = helpers.DataInterface( df , continuous_features = continuous_features)

 ##model_interface = helpers.ModelInterface(xgb_classifier)

# # Step 1: Initialize DiCE

 ##d = Dice(d, data_interface=data_interface, model_interface=model_interface)
from dice_ml import Dice

# Step 1: Initialize DiCE
d = Dice(dataframe=df, continuous_features=continuous_features, outcome_name=outcome_name, model=xgb_classifier)




NameError: name 'xgb_classifier' is not defined